# Modeling
- 미션 10의 `News Group 20 뉴스 기사 카테고리 분류 LSTM 모델`을 로드 및 학습합니다.
- 모델을 `.pth` 형식으로 저장합니다.
- 양자화을 진행한 후 `.pth` 형식으로 저장합니다.
- 모델을 `.onnx` 형식으로 내보냅니다.


## 1. News Group 20 뉴스 기사 카테고리 분류 LSTM 모델

### 1-1. 환경 설정 및 데이터 준비

In [27]:
!pip install gensim

In [28]:
# 필요한 라이브러리 임포트
import os
from gensim.models import FastText
import torch
import torch.nn as nn
from torch.utils.data import Dataset
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import nltk

# NLTK 리소스 다운로드
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import matplotlib.pyplot as plt

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [29]:
# GPU 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


### 1-2. 데이터 로드 및 전처리


In [30]:
from sklearn.datasets import fetch_20newsgroups

news_data = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))

texts = news_data.data
labels = news_data.target

In [57]:
import json

# 학습 데이터셋에 들어있던 레이블 리스트
target_names_list = list(news_data.target_names)

# JSON 파일로 저장 (순서가 그대로 보존됩니다)
with open("./models/label_names.json", "w") as f:
    json.dump(target_names_list, f)

### 1-3. 데이터 분할 및 불용어 제거

In [31]:
# 학습/테스트 분할
train_inputs, test_inputs, train_targets, test_targets = train_test_split(texts, labels, test_size=0.2, random_state=42)

In [32]:
import re

nltk.download('stopwords')
stop_words = stopwords.words('english')

# 텍스트 데이터 전처리
def clean_text(text):
    """
    텍스트를 소문자로 변환하고,
    특수문자 제거 및 불용어 제거를 수행한 뒤
    정제된 텍스트를 반환
    """
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    stop_words = set(stopwords.words('english'))
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(tokens)

# 전처리 적용
train_inputs = [clean_text(text) for text in train_inputs]
test_inputs = [clean_text(text) for text in test_inputs]

# 단어 토큰화 진행
train_sentences = [word_tokenize(text) for text in train_inputs]
test_sentences = [word_tokenize(text) for text in test_inputs]

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [33]:
max_len = 280

### 1-4. 임베딩 적용 및 텍스트 분류 모델 구현

In [34]:
class TextEmbeddingDataset(Dataset):
  def __init__(self, texts, labels, word2idx, max_len):
    self.texts = texts
    self.labels = labels
    self.word2idx = word2idx
    self.max_len = max_len

  def __len__(self):
    return len(self.texts)

  def __getitem__(self,idx):
    tokens = word_tokenize(self.texts[idx])

    encoded = [self.word2idx.get(word, 0) for word in tokens]

    if len(encoded) < self.max_len:
      encoded += [0] * (self.max_len - len(encoded))
    else:
      encoded = encoded[:self.max_len]

    return torch.tensor(encoded, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

In [35]:
# FastText 모델 학습
fasttext_model = FastText(
    sentences=train_sentences,
    vector_size=128,
    window=5,
    min_count=1,
    sg=1)

# 임베딩 행렬 초기화
fasttext_matrix = np.zeros((len(fasttext_model.wv) + 1, 128))

# 단어 -> 인덱스 매핑 생성
word2idx_fasttext = {word: idx + 1 for idx, word in enumerate(fasttext_model.wv.index_to_key)}

# 임베딩 행렬에 FastText 벡터 채우기
for word, idx in word2idx_fasttext.items():
    fasttext_matrix[idx] = fasttext_model.wv[word]

In [47]:
# 추후 추론을 위한 단어장 저장
import pickle
import os

os.makedirs("./models", exist_ok=True)
dict_save_path = "./models/word2idx_fasttext.pkl"

# 단어장(dict)을 'wb'(Write Binary) 모드로 저장합니다.
with open(dict_save_path, "wb") as f:
    pickle.dump(word2idx_fasttext, f)

print(f"단어장 저장 완료: {dict_save_path}")

단어장 저장 완료: ./models/word2idx_fasttext.pkl


### 1-5. LSTM 기반 분류 모델 구조 정의

In [36]:
class EmbeddingLSTM(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim, output_dim, num_layers=2, dropout=0.5):
        super(EmbeddingLSTM, self).__init__()

        num_embeddings, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float).to(device),
            freeze=True)

        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout, bidirectional=True)

        self.fc = nn.Linear(hidden_dim, output_dim)


    def forward(self, x):
        embedded = self.embedding(x)

        _, (hidden, _) = self.lstm(embedded)

        output = self.fc(hidden[-1])

        return output

### 1-6. 학습 설정 및 모델 학습

In [37]:
def train(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    for texts, labels in loader:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(texts)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [38]:
# FastText Dataset
train_dataset_fasttext = TextEmbeddingDataset(train_inputs, train_targets, word2idx_fasttext, max_len)
test_dataset_fasttext = TextEmbeddingDataset(test_inputs, test_targets, word2idx_fasttext, max_len)

# FastText DataLoader
train_loader_fasttext = DataLoader(train_dataset_fasttext, batch_size=64, shuffle=True)
test_loader_fasttext = DataLoader(test_dataset_fasttext, batch_size=64, shuffle=False)

In [39]:
hidden_dim = 128
output_dim = len(set(labels))

loss_fn = nn.CrossEntropyLoss()
lr = 0.005
num_epochs = 10
batch_size = 64

In [40]:
os.makedirs("./models", exist_ok=True)

# FastText 학습 루프
model_fasttext = EmbeddingLSTM(fasttext_matrix, hidden_dim, output_dim).to(device)
optimizer = torch.optim.Adam(model_fasttext.parameters(), lr=lr)

for epoch in range(10):
    loss = train(model_fasttext, train_loader_fasttext, loss_fn, optimizer)
    print(f"Epoch {epoch+1}, Loss: {loss:.4f}")

print("학습이 완료되었습니다.")

Epoch 1, Loss: 2.2227
Epoch 2, Loss: 1.7263
Epoch 3, Loss: 1.4337
Epoch 4, Loss: 1.3125
Epoch 5, Loss: 1.2102
Epoch 6, Loss: 1.1375
Epoch 7, Loss: 1.0708
Epoch 8, Loss: 1.0231
Epoch 9, Loss: 0.9569
Epoch 10, Loss: 0.9082
학습이 완료되었습니다.


## 2. 모델 저장

In [41]:
!pip install onnx onnxscript

### 2-1. 학습한 모델을 `.pth`형식으로 저장

In [42]:
# 1. 표준 PyTorch 모델 저장 (.pth)
base_save_path = "./models/mission_10_lstm_model.pth"
torch.save(model_fasttext.state_dict(), base_save_path)
print(f"1. 표준 PyTorch 모델 저장 완료: {base_save_path}")

1. 표준 PyTorch 모델 저장 완료: ./models/mission_10_lstm_model.pth


### 2-2. 모델 양자화를 진행 후 `.pth`형식으로 저장
- 동적 양자화 적용: 모델의 가중치를 FP32(PyTorch 모델 기본값)에서 INT8로 압축하여 용량 감소 및 연산 속도 향상.

In [43]:
# 2. 동적 양자화 진행 및 저장 (.pth)
import torch.onnx

model_cpu = model_fasttext.to('cpu') # 양자화 추론을 위해 모델을 CPU로 이동
quantized_model = torch.quantization.quantize_dynamic(
    model_cpu,
    {torch.nn.LSTM, torch.nn.Linear},  # LSTM과 Linear 레이어 타겟팅
    dtype=torch.qint8
)
quantized_save_path = "./models/mission_10_lstm_quantized.pth"
torch.save(quantized_model.state_dict(), quantized_save_path)
print(f"2. 양자화 모델 저장 완료: {quantized_save_path}")


2. 양자화 모델 저장 완료: ./models/mission_10_lstm_quantized.pth


/tmp/ipykernel_615/1291731472.py:5: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


### 2-3. 학습한 모델을 `.onnx` 형식으로 변환

In [44]:
# 3. ONNX 포맷으로 내보내기 (.onnx)
import torch.onnx

onnx_save_path = "./models/mission_10_lstm_model.onnx"

# ONNX로 내보내려면 모델이 데이터를 어떻게 받는지 알려줄 '가짜 입력(Dummy Input)'이 필요합니다.
dummy_input = torch.randint(0, len(word2idx_fasttext), (1, max_len)).to('cpu')

# 모델을 평가(추론) 모드로 변경 (Dropout 등을 끔)
model_fasttext.eval()
import torch.onnx
torch.onnx.export(
    model_fasttext,
    dummy_input,
    onnx_save_path,
    export_params=True,           # 모델의 가중치(파라미터)도 함께 저장
    opset_version=11,             # 보편적으로 안정적인 ONNX 버전
    do_constant_folding=True,     # 상수 폴딩 최적화
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
    'input': {0: 'batch_size'},
    'output': {0: 'batch_size'}
}
)
print(f"3. ONNX 모델 내보내기 완료: {onnx_save_path}")

/tmp/ipykernel_615/803566054.py:12: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0317 06:08:02.834000 615 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `EmbeddingLSTM([...]` with `torch.export.export(..., strict=False)`...


/usr/lib/python3.12/contextlib.py:144: UserWarning: The tensor attributes self.lstm._flat_weights[0], self.lstm._flat_weights[1], self.lstm._flat_weights[2], self.lstm._flat_weights[3], self.lstm._flat_weights[4], self.lstm._flat_weights[5], self.lstm._flat_weights[6], self.lstm._flat_weights[7], self.lstm._flat_weights[8], self.lstm._flat_weights[9], self.lstm._flat_weights[10], self.lstm._flat_weights[11], self.lstm._flat_weights[12], self.lstm._flat_weights[13], self.lstm._flat_weights[14], self.lstm._flat_weights[15] were assigned during export. Such attributes must be registered as buffers using the `register_buffer` API (https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer).
  next(self.gen)


[torch.onnx] Obtain model graph for `EmbeddingLSTM([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 2 of general pattern rewrite rules.
3. ONNX 모델 내보내기 완료: ./models/mission_10_lstm_model.onnx


- Troubleshooting: ONNX 변환 시 dynamic_axes에 sequence_length를 가변(Dynamic)으로 설정했을 때 Shape 불일치 에러 발생 (특히 LSTM 구조에서 자주 발생).

In [58]:
# 파일 용량 출력
def get_file_size(file_path):
    size = os.path.getsize(file_path) / (1024 * 1024)
    return f"{size:.2f} MB"

print(f"표준 PTH 모델 크기: {get_file_size(base_save_path)}")
print(f"양자화된 PTH 모델 크기: {get_file_size(quantized_save_path)}")
print(f"ONNX 모델 크기: {get_file_size(onnx_save_path)}")

표준 PTH 모델 크기: 52.60 MB
양자화된 PTH 모델 크기: 50.73 MB
ONNX 모델 크기: 0.09 MB
